In [ ]:
GPU="0"
num_GPUs = 1
gen_image_batch_size=512

In [ ]:
import os
os.environ['PATH'] += ':/home/temp0/anaconda3/envs/edm2/bin'

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

def show_images_from_dir(folder_path, num_images=4, start_seed=42):
    """
    顯示指定資料夾中的圖片（根據檔名排序，從指定種子碼開始），以 2x2 格顯示。

    Args:
        folder_path (str): 圖片所在資料夾路徑
        num_images (int): 要顯示的圖片數量（預設 4）
        start_seed (int): 從第幾個種子（圖片）開始（根據檔名排序）
    """
    if not os.path.isdir(folder_path):
        print(f"[錯誤] 資料夾不存在：{folder_path}")
        return

    # 過濾圖片檔案，並根據檔名中的數字排序
    image_files = sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ], key=lambda x: int(''.join(filter(str.isdigit, os.path.basename(x))) or 0))

    # 找到起始 index
    start_index = next((i for i, f in enumerate(image_files)
                        if int(''.join(filter(str.isdigit, os.path.basename(f))) or 0) >= start_seed), None)

    if start_index is None:
        print(f"[警告] 找不到 seed >= {start_seed} 的圖片。")
        return

    subset = image_files[start_index:start_index + num_images]
    if not subset:
        print(f"[警告] 從 seed {start_seed} 起沒有足夠圖片可顯示。")
        return

    # 顯示 2x2 圖片
    rows = cols = 2
    plt.figure(figsize=(8, 8))
    for i, img_path in enumerate(subset):
        img = Image.open(img_path)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# S 5萬 NoGuid

In [ ]:
k=2
w=3.0
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 19
w_interval_high_middle_steps = 21
w_interval_high_steps = 30
num_images = 50000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_noguid_const_w={w}_k={k},polygon,{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=noguid \
--guidance_scheduler=const_scheduler \
--debug=False \
--guidance={w} \
--k_average={k} \
--full_random=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
k=2
w=3.0
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 19
w_interval_high_middle_steps = 21
w_interval_high_steps = 30
num_images = 50000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_noguid_const_w={w}_k={k},polygon,{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG 5萬用錯W=1.7

In [ ]:
k=2
w=1.7
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 19
w_interval_high_middle_steps = 21
w_interval_high_steps = 30
num_images = 50000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_const_w={w}_k={k},polygon,{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=const_scheduler \
--debug=False \
--guidance={w} \
--k_average={k} \
--full_random=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG 5萬用錯W=1.7

In [ ]:
k=2
w=1.7
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 19
w_interval_high_middle_steps = 21
w_interval_high_steps = 30
num_images = 50000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_const_w={w}_k={k},polygon,{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 512 Autoguid S

In [ ]:
k=2
w=2.1
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 19
w_interval_high_middle_steps = 21
w_interval_high_steps = 30
num_images = 50000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_autoguid_const_w={w}_k={k},polygon,{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-autog-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=const_scheduler \
--debug=False \
--guidance={w} \
--k_average={k} \
--full_random=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 重作CFG const 因為原論文使用w=1.4


In [ ]:
w=1.4
num_images = 50000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_const_w={w}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=const_scheduler \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 原論文 Interval CFG

In [ ]:
w=2.1
num_images = 50000
w_interval_low_sigma=0.28
w_interval_high_sigma=2.9
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_interval_scheduler_w={w},w_interval_low_sigma={w_interval_low_sigma},w_interval_high_sigma={w_interval_high_sigma}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=interval_scheduler \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_sigma={w_interval_low_sigma} \
--w_interval_high_sigma={w_interval_high_sigma} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# Optimal RN:polygon(14,15,18,24,32) w=2.1

In [ ]:
w=2.1
num_images = 50000
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 32
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_RN_polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=random_ddg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=False \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# Optimal RN Trial on Autoguid:polygon(14,15,18,24,32) w=2.1

In [ ]:
w=2.1
num_images = 50000
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 32
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_RN_autoguid_polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-autog-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=random_ddg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=False \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 測試 對CFG進行schedule

# BaseLine CFG const w=1.4 1萬張

In [ ]:
w=1.4
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_const_w={w}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=const_scheduler \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=2.1 1萬張

In [ ]:
w=2.1
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_const_w={w},,polygon,{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.9 1萬張

In [ ]:
w=1.9
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.7 1萬張

In [ ]:
w=1.7
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.9 1萬張 前段至12

In [ ]:
w=1.9
w_interval_low_steps = 12
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.9 1萬張 前段至10

In [ ]:
w=1.9
w_interval_low_steps = 10
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.9 1萬張 後段至32

In [ ]:
w=1.9
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 24
w_interval_high_steps = 32
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.9 1萬張 middle=26

In [ ]:
w=1.9
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 26
w_interval_high_steps = 32
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.9 1萬張 middle=28

In [ ]:
w=1.9
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 28
w_interval_high_steps = 32
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=2.0 1萬張 middle=28,end=30

In [ ]:
w=2.0
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=2.0 1萬張 high_peek=20,middle=28,end=30

In [ ]:
w=2.0
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 20
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=2.0 1萬張 (15,16),20,28,30

In [ ]:
w=2.0
w_interval_low_steps = 15
w_interval_low_peak_steps = 16
w_interval_high_peak_steps = 20
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.95 1萬張 (15,16,18,28,30)

In [ ]:
w=1.95
w_interval_low_steps = 15
w_interval_low_peak_steps = 16
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.95 1萬張 (16,17,18,28,30)

In [ ]:
w=1.95
w_interval_low_steps = 16
w_interval_low_peak_steps = 17
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.95 1萬張 (14,17,18,28,30)

In [ ]:
w=1.95
w_interval_low_steps = 14
w_interval_low_peak_steps = 17
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.95 1萬張 (14,12,18,28,30)

In [ ]:
w=1.95
w_interval_low_steps = 12
w_interval_low_peak_steps = 17
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.95 1萬張 (16,19,20,28,30)

In [ ]:
w=1.95
w_interval_low_steps = 16
w_interval_low_peak_steps = 19
w_interval_high_peak_steps = 20
w_interval_high_middle_steps = 28
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# CFG polygon w=1.95 1萬張 (16,19,20,26,30)

In [ ]:
w=1.95
w_interval_low_steps = 16
w_interval_low_peak_steps = 19
w_interval_high_peak_steps = 20
w_interval_high_middle_steps = 26
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=1.95
w_interval_low_steps = 16
w_interval_low_peak_steps = 19
w_interval_high_peak_steps = 20
w_interval_high_middle_steps = 26
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0821_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=1.95
w_interval_low_steps = 16
w_interval_low_peak_steps = 19
w_interval_high_peak_steps = 20
w_interval_high_middle_steps = 26
w_interval_high_steps = 30
num_images = 10000
# gen_image_batch_size = 32
#--guidance_scheduler=polygon_schedule \
dirname=f'/data/guidance-team/out/2025_0821_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=1.95
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 26
w_interval_high_steps = 30
num_images = 50000

dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.0
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 18
w_interval_high_middle_steps = 26
w_interval_high_steps = 30
num_images = 50000

dirname=f'/data/guidance-team/out/2025_0820_cfg_10000_,polygon_w={w},{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_middle_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=polygon_schedule \
--debug=False \
--guidance={w} \
--heun_guid=True \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_middle_steps={w_interval_high_middle_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000